# 🧠 Introverts vs Extroverts — Kişilik Sınıflandırma Projesi

## 🎯 Problem
Bireylerin sosyal alışkanlıkları ve davranışsal özelliklerinden yararlanarak kişilik tipinin **İçe dönük (`Introvert`)** mü yoksa **Dışa dönük (`Extrovert`)** mü olduğunu tahmin ediyoruz.

## 💡 Bu Notebook'tan Ne Öğreneceksin?
- `Pipeline` ve `ColumnTransformer` kullanarak eksik değer doldurma ve kategorik kodlamayı tek çatı altında yönetmeyi
- Sayısal (`median`) ve kategorik (`most_frequent`) eksik değer doldurma stratejilerini bir arada kullanmayı
- `OneHotEncoder(handle_unknown="ignore")` ile eğitimde görülmeyen kategorilere karşı dayanıklılık sağlamayı
- Yalnızca tek bir train/test bölünmesine güvenmek yerine **5-Fold Cross-Validation** ile kararlılığı ölçmeyi
- Accuracy yanında **F1-Score**, **ROC-AUC** ve **Classification Report** ile modeli kapsamlı değerlendirmeyi

## 📊 Veri Seti
- **Kaynak:** [Kaggle - Playground Series S5E7](https://www.kaggle.com/competitions/playground-series-s5e7)
- **Özellikler:** `Time_spent_Alone`, `Stage_fear`, `Social_event_attendance`, `Going_outside`, `Drained_after_socializing`, `Friends_circle_size`, `Post_frequency`
- **Hedef Değişken:** `Personality` (`Extrovert` / `Introvert`)

## 1. Kütüphaneler ve Veri Yükleme

Temel kütüphaneleri içe aktarıyor, Kaggle yarışma dizinindeki `train.csv` ve `test.csv` dosyalarını yüklüyoruz.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s5e7/sample_submission.csv
/kaggle/input/competitions/playground-series-s5e7/train.csv
/kaggle/input/competitions/playground-series-s5e7/test.csv


In [2]:
train=pd.read_csv((__import__("glob").glob("/kaggle/input/**/train.csv", recursive=True)[0] if not os.path.exists("/kaggle/input/playground-series-s5e7/train.csv") else "/kaggle/input/playground-series-s5e7/train.csv"))
train.head()

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,0,0.0,No,6.0,4.0,No,15.0,5.0,Extrovert
1,1,1.0,No,7.0,3.0,No,10.0,8.0,Extrovert
2,2,6.0,Yes,1.0,0.0,NaN,3.0,0.0,Introvert
3,3,3.0,No,7.0,3.0,No,11.0,5.0,Extrovert
4,4,1.0,No,4.0,4.0,No,13.0,NaN,Extrovert


In [3]:
test=pd.read_csv((__import__("glob").glob("/kaggle/input/**/test.csv", recursive=True)[0] if not os.path.exists("/kaggle/input/playground-series-s5e7/test.csv") else "/kaggle/input/playground-series-s5e7/test.csv"))

## 2. Veri Temizliği ve Keşifçi Veri Analizi (EDA)

Kimlik sütunu olan `id` modellemeye katkı sağlamayacağı için çıkarılır. Kategorik sütunlardaki benzersiz değerler (`Stage_fear`) ve tüm sütunlardaki eksik veri (`NaN`) miktarları incelenir.

In [4]:
train=train.drop("id",axis=1)

In [5]:
train["Stage_fear"].unique()

array(['No', 'Yes', nan], dtype=object)

In [6]:
train.isnull().sum()

Time_spent_Alone             1190
Stage_fear                   1893
Social_event_attendance      1180
Going_outside                1466
Drained_after_socializing    1149
Friends_circle_size          1054
Post_frequency               1264
Personality                     0
dtype: int64

## 3. Train-Test Bölünmesi ve Ön İşleme Pipeline'ı

Hedef değişken (`Personality`) ile özellikleri ayırıp veriyi %80 eğitim / %20 test olarak bölüyoruz.
- **Sayısal sütunlar:** Eksik değerler aykırı değerlere dayanıklı olması için `median` ile doldurulur.
- **Kategorik sütunlar:** Eksik değerler `most_frequent` ile doldurulur ve `OneHotEncoder(handle_unknown="ignore")` uygulanır.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

y = train["Personality"]
x = train.drop("Personality", axis=1)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

num_cols = list(x_train.select_dtypes(exclude="object").columns)
cat_cols = list(x_train.select_dtypes(include="object").columns)

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

## 4. Model Eğitimi (`RandomForestClassifier`) ve İlk Skorlar

Ön işleme adımıyla `RandomForestClassifier(n_estimators=150, max_depth=4)` modelini bir `Pipeline` içinde birleştirip eğitiyoruz. Ardından eğitim ve test accuracy skorlarını karşılaştırarak aşırı öğrenme (overfitting) olup olmadığını kontrol ediyoruz.

In [8]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=150, max_depth=4))
])

model.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Time_spent_Alone',
                                                   'Social_event_attendance',
                                                   'Going_outside',
                                                   'Friends_circle_size',
                                                   'Post_frequency']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Stage_fear',
                                                   'Drained_after_socializing'])])),
                ('model',
                 RandomForestClassifier(max_depth=4, n_estimators=150))])

In [9]:
print(model.score(x_train,y_train))
print(model.score(x_test,y_test))

0.9692286928942574
0.9686909581646423


## 5. Çapraz Doğrulama (5-Fold Cross-Validation)

Tek bir `train_test_split` rastgele bölünmeye bağlı şans faktörü içerebilir. Modelin farklı veri alt kümelerindeki kararlılığını ölçmek için `x_train` üzerinde **5 katlı çapraz doğrulama (5-Fold CV)** ile F1 skorunun ortalamasını ve standart sapmasını hesaplıyoruz.

> **Not:** Hedef değişken string etiketler (`'Extrovert'`, `'Introvert'`) içerdiği için `make_scorer(f1_score, pos_label='Extrovert')` kullanılarak pozitif sınıf açıkça belirtilir.

In [10]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, make_scorer

f1_scorer = make_scorer(f1_score, pos_label='Extrovert')
cv_scores = cross_val_score(model, x_train, y_train, cv=5, scoring=f1_scorer)
print(f"5-Fold CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

5-Fold CV F1: 0.9787 ± 0.0023


## 6. Detaylı Performans Değerlendirmesi: F1-Score, ROC-AUC ve Classification Report

Yalnızca doğruluk (accuracy) metriği sınıf bazlı hataları gizleyebilir. Bu nedenle test seti üzerinde:
- **F1 Score:** Precision ve Recall dengesini ölçer.
- **ROC-AUC:** Modelin olasılık sıralama (ayırt etme) gücünü ölçer (`model.classes_` içinden `'Extrovert'` sınıfının olasılık indeksi seçilerek hesaplanır).
- **Classification Report:** Her iki sınıf (`Extrovert` ve `Introvert`) için ayrıntılı precision, recall ve F1 değerlerini sunar.

In [11]:
from sklearn.metrics import f1_score, roc_auc_score, classification_report
y_pred = model.predict(x_test)
extrovert_idx = list(model.classes_).index('Extrovert')
y_prob = model.predict_proba(x_test)[:, extrovert_idx]
print(f"F1 Score: {f1_score(y_test, y_pred, pos_label='Extrovert'):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test == 'Extrovert', y_prob):.4f}")
print(classification_report(y_test, y_pred))

F1 Score: 0.9790
ROC-AUC: 0.9649
              precision    recall  f1-score   support

   Extrovert       0.98      0.98      0.98      2753
   Introvert       0.95      0.93      0.94       952

    accuracy                           0.97      3705
   macro avg       0.96      0.96      0.96      3705
weighted avg       0.97      0.97      0.97      3705



## 7. Kaggle Submission Dosyası Oluşturma

Eğitilmiş `model` pipeline'ı ile `test` veri seti için kişilik tahminlerini üretip `submission.csv` olarak kaydediyoruz.

In [12]:
pred=model.predict(test)

In [13]:
submission=pd.read_csv((__import__("glob").glob("/kaggle/input/**/sample_submission.csv", recursive=True)[0] if not os.path.exists("/kaggle/input/playground-series-s5e7/sample_submission.csv") else "/kaggle/input/playground-series-s5e7/sample_submission.csv"))
submission.head()

,id,Personality
0,18524,Extrovert
1,18525,Extrovert
2,18526,Extrovert
3,18527,Extrovert
4,18528,Extrovert


In [14]:
submission["Personality"]=pred

submission.to_csv("submission.csv", index=False)

## 📈 Özet ve Öğrenilenler

### ✅ Yapılan İyileştirmeler
1. **5-Fold Cross-Validation eklendi:** Modelin yalnızca tek bir split'teki skoru değil, 5 farklı katmandaki ortalama F1 skoru ve standart sapması için doğrulama hücresi eklendi *(çıktılar Run All ile üretilir)*.
2. **Kapsamlı değerlendirme metrikleri eklendi:** Sadece `model.score()` (accuracy) yerine `F1 Score`, `ROC-AUC` ve `classification_report` hesaplama hücresi eklendi *(çıktılar Run All ile üretilir)*.
3. **Türkçe dokümantasyon:** Tüm adımlar anlaşılır Türkçe açıklamalarla düzenlendi.

### 🔑 Öğrenilenler
- Eğitim (%96.92) ve test (%96.87) doğruluk skorlarının birbirine çok yakın olması `max_depth=4` kısıtı sayesinde modelin aşırı öğrenmediğini gösterir.
- `Pipeline` içinde `SimpleImputer` ve `OneHotEncoder` kullanmak, yarışma `test` verisinde ön işleme hatalarını ve veri sızıntısını önler.
- String sınıf etiketleriyle (`'Extrovert'`, `'Introvert'`) çalışırken `pos_label` ve `model.classes_` sırasına dikkat etmek metrik hesaplama hatalarının önüne geçer.